<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/02_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/yaranoun/ML-Tech.git

Cloning into 'ML-Tech'...
remote: Enumerating objects: 242, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 242 (delta 29), reused 11 (delta 4), pack-reused 190 (from 1)
Receiving objects: 100% (242/242), 193.13 KiB | 2.84 MiB/s, done.
Resolving deltas: 100% (133/133), done.


In [ ]:
%cd /content/ML-Tech
!git pull origin main

/content/ML-Tech
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Already up to date.


In [ ]:
import json

# each line of the bundled file is one {"user", "assistant"} chat example
with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"loaded {len(chunks)} chunks")
print(chunks[0])

loaded 19 chunks
{'document': 'Personal attendance required.txt', 'title': 'Personal attendance required', 'url': 'https://www.general-security.gov.lb/en/posts/73', 'category': 'Personal attendance required', 'keywords': 'Lebanese citizens, minors, exemption from attendance, exemption from fees', 'section': 'Personal attendance required', 'text': 'Lebanese citizens that request a new passport should show up personally at the competent regional center of general security, according to their place of residence, having in hand an application that’s been filled, and certified by the competent mayor.\nMinors aged 7 years or younger have to accompany their parents to the mayor’s office, but don’t have to show up at the general security center. Both parents should sign a letter of consent at the mayor’s office, and convey their request to the general security. One of the parents can go on his own to the general security office, if the other parent signed the letter at the mayor’s office.\nMin

In [ ]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.5 MB/s eta 0:00:00


In [32]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("intfloat/multilingual-e5-base")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [33]:
passages = [
    "passage: " + chunk["section"] + "\n" + chunk["text"]
    for chunk in chunks
]

In [34]:
embeddings = embedding_model.encode(
    passages,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(19, 768)


In [35]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
faiss.write_index(index, "data/processed/passport_index.faiss")

In [41]:
faiss.write_index(
    index,
    "data/processed/passport_index.faiss"
)

In [37]:
import os

file_path = "data/processed/passport_index.faiss"
if os.path.exists(file_path):
    print(f"The file '{file_path}' exists.")
else:
    print(f"Error: The file '{file_path}' does not exist. Please re-run the cell that creates it (cell `CMhSIbmozWxF`).")

The file 'data/processed/passport_index.faiss' exists.


In [38]:
import faiss

# Load the index to confirm it was created successfully
loaded_index = faiss.read_index("data/processed/passport_index.faiss")
print(f"Loaded FAISS index with {loaded_index.ntotal} vectors and dimension {loaded_index.d}.")

Loaded FAISS index with 19 vectors and dimension 768.


In [39]:
def retrieve(question, k=3):
    query_embedding = embedding_model.encode(
        ["query: " + question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "document": chunks[idx]["document"],
            "section": chunks[idx]["section"],
            "text": chunks[idx]["text"]
        })

    return results

In [40]:
results = retrieve("What are the fees to get a biometric passport?")

for result in results:
    print(result["score"])
    print(result["document"])
    print(result["section"])
    print(result["text"])
    print()

0.848317563533783
Biometric Passport.txt
Requested documents
The adequate application for passports format A4 (10 years) issued by the competent mayor according to the place of residence.
Lebanese ID card OR/AND an extract of civil status (whether the Lebanese citizen is applying for the 1st time for a biometric passport or not). Follow this link for more information: https://www.general-security.gov.lb/ar/posts/408
A new colored photo ID photo on a white background, 4.5 x 3.5, on which the name of the individual appears, as well as the number and place of registered residence, signed and certified by the mayor.
The old passport if the latter is available, as well as a copy of the pages that are not empty.
The fees related to this application.
When it comes to members of the general security, whether active or retired, and the applications sent by their families, they can hand out only one document of identification (identity card, or extract of individual civil status, which dates bac

In [ ]:
!git config --global user.email "yarajnoun@gmail.com"
!git config --global user.name "yaranoun"

In [ ]:
!git add data/processed/passport_index.faiss
!git commit -m "Update document FAISS"
!git pull --rebase origin main
!git push origin main

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Current branch main is up to date.
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
%cd /content/ML-Tech
!git pull origin main

/content/ML-Tech
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Already up to date.


In [ ]:
!ls data/processed

chunks.json  passport_index.faiss
